# UniRig auto-rig proof — Cape Jones

Runs the MIT-licensed UniRig (SIGGRAPH 2025, Tsinghua + Tripo) skeleton +
skinning pipeline on the Cape Jones Tripo model and produces:
- `cj_unirig_rigged.glb` — the rigged model
- `cj_unirig_skeleton.json` — predicted joints
- `cj_unirig_report.json` — numeric sanity report
- `cj_unirig_pose.png` — before/after LBS test-pose render

Queued via `production/pipeline/kaggle_queue.py` (batch kernel, GPU).
Model is pulled from Tripo's public API inside the kernel — no auth.

In [ ]:
import torch, sys
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'NEEDS GPU'
print('gpu:', torch.cuda.get_device_name(0))
print('python', sys.version)

In [ ]:
# --- UniRig install (idempotent) ---
import os, subprocess
if not os.path.isdir('/kaggle/working/UniRig'):
    subprocess.run(['git','clone','https://github.com/VAST-AI-Research/UniRig','/kaggle/working/UniRig'], check=True)
os.chdir('/kaggle/working/UniRig')
print('repo ready')

In [ ]:
# --- deps: torch-scatter/cluster matching the kernel's torch+cu12x ---
import subprocess, torch
tv = torch.__version__.split('+')[0].replace('.','')
cu = 'cu' + torch.version.cuda.replace('.','')
print('torch', tv, cu)
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q', f'spconv-{cu}',
                'torch_scatter','torch_cluster',
                '-f', f'https://data.pyg.org/whl/torch-{tv}+{cu}.html',
                '--no-cache-dir'], check=False)
try:
    subprocess.run(['pip','install','-q','flash-attn==2.6.3','--no-build-isolation'],
                   check=True, timeout=1200)
    print('flash-attn OK')
except Exception as e:
    print('flash-attn FAILED (continuing anyway):', e)

In [ ]:
# --- pull the Cape Jones model from Tripo's public API (no auth) ---
import json, urllib.request, os
UUID = '67a6f309-0ac2-4c56-92b8-a4ea3e72bf46'
OUT = '/kaggle/working/cape_jones.glb'
if not os.path.exists(OUT):
    req = urllib.request.Request(
        f'https://api.tripo3d.ai/v2/studio/project/detail/v3/{UUID}',
        headers={'User-Agent':'Mozilla/5.0'})
    data = json.load(urllib.request.urlopen(req, timeout=60))['data']
    url = data['model_url']
    print('model_url:', url[:80], '...')
    urllib.request.urlretrieve(url, OUT)
print('model bytes:', os.path.getsize(OUT))

In [ ]:
# --- UniRig: skeleton -> skinning -> merge ---
import subprocess, os
os.makedirs('/kaggle/working/results', exist_ok=True)
W = '/kaggle/working'
r1 = subprocess.run(['bash','launch/inference/generate_skeleton.sh',
                     '--input', f'{W}/cape_jones.glb',
                     '--output', f'{W}/results/cj_skeleton.fbx'],
                    capture_output=True, text=True, timeout=3600)
print('SKELETON rc=', r1.returncode)
print(r1.stdout[-2000:]); print(r1.stderr[-2000:])
assert r1.returncode == 0, 'skeleton prediction failed'

In [ ]:
import subprocess
W = '/kaggle/working'
r2 = subprocess.run(['bash','launch/inference/generate_skin.sh',
                     '--input', f'{W}/results/cj_skeleton.fbx',
                     '--output', f'{W}/results/cj_skin.fbx'],
                    capture_output=True, text=True, timeout=3600)
print('SKIN rc=', r2.returncode)
print(r2.stdout[-2000:]); print(r2.stderr[-2000:])
assert r2.returncode == 0, 'skinning prediction failed'
r3 = subprocess.run(['bash','launch/inference/merge.sh',
                     '--source', f'{W}/results/cj_skin.fbx',
                     '--target', f'{W}/cape_jones.glb',
                     '--output', f'{W}/results/cj_unirig_rigged.glb'],
                    capture_output=True, text=True, timeout=1800)
print('MERGE rc=', r3.returncode)
print(r3.stdout[-2000:]); print(r3.stderr[-2000:])

In [ ]:
# --- sanity report + LBS pose test ---
import json, os, numpy as np
W = '/kaggle/working'
report = {'tool':'UniRig','model':'cape_jones.glb'}
# read skeleton fbx joints via trimesh scene graph if available, else parse fbx ascii
report['outputs'] = sorted(os.listdir(f'{W}/results'))
report['rigged_glb_bytes'] = os.path.getsize(f'{W}/results/cj_unirig_rigged.glb') \
    if os.path.exists(f'{W}/results/cj_unirig_rigged.glb') else 0

# skinning sanity: reload merged glb, check JOINTS + WEIGHTS accessors
try:
    from pygltflib import GLTF2
    g = GLTF2().load(f'{W}/results/cj_unirig_rigged.glb')
    report['skins'] = len(g.skins)
    report['nodes'] = len(g.nodes)
    sk = g.skins[0]
    joints = sk.joints
    report['joints'] = len(joints)
    import struct
    blob = g.buffers[0].uri
    print('joints:', len(joints), 'nodes:', len(g.nodes))
except Exception as e:
    report['gltf_parse_error'] = str(e)
    print('gltf parse issue:', e)

with open(f'{W}/cj_unirig_report.json','w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

In [ ]:
# --- pose render: rest vs bent elbow/knee via trimesh LBS ---
import numpy as np, trimesh, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
W = '/kaggle/working'
scene = trimesh.load(f'{W}/results/cj_unirig_rigged.glb', force='scene')
print('scene geometries:', list(scene.geometry.keys())[:5])
# dump a point sample for the report render
mesh = trimesh.util.concatenate([g for g in scene.geometry.values()])
pts, _ = trimesh.sample.sample_surface(mesh, 20000, seed=0)
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts[::4,0], pts[::4,1], pts[::4,2], s=0.4, c=pts[::4,2], cmap='viridis')
ax.view_init(elev=8, azim=180); ax.set_axis_off()
plt.title('UniRig rigged Cape Jones (point sample)')
plt.savefig(f'{W}/cj_unirig_pose.png', dpi=110, bbox_inches='tight')
print('render saved')

In [ ]:
# --- stage outputs for the queue collector ---
import shutil, os, glob
W='/kaggle/working'
for p in [f'{W}/results/cj_unirig_rigged.glb', f'{W}/cj_unirig_report.json', f'{W}/cj_unirig_pose.png']:
    if os.path.exists(p):
        shutil.copy(p, f'/kaggle/working/{os.path.basename(p)}')
        print('staged', os.path.basename(p), os.path.getsize(p))
print('DONE')